# Module 20: Capstone setup

This module takes about an hour - largely because we have a lot of tables and views and Data Insights API takes some time to execute scans even if run parallely. It is critical to finish each step all the way to step 5.2 which needs to complete successfully to get started with custom agents.

## 0.0. Authenticate, Install packages, Variables, Configs

In [ ]:
!gcloud auth application-default login

In [ ]:
!pip install google-cloud-dataplex==2.11.0  -q

In [ ]:
PROJECT_ID= ! gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
PROJECT_NAME= ! gcloud projects describe $PROJECT_ID | grep name | cut -d':' -f2 | xargs
PROJECT_NAME=PROJECT_NAME[0]

LOCATION="us-central1"
DATA_SCAN_API_CREATE_ENDPOINT_PREFIX=f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId="
DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX=(f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/")
SCOPES = ['https://www.googleapis.com/auth/cloud-platform']
BASE_URL_FOR_DATAPLEX_SCAN="https://dataplex.googleapis.com/v1"


DATA_INGESTION_BUCKET=f"capstone_stage_{PROJECT_NBR}"
METADATA_BUCKET=f"capstone_stage_{PROJECT_NBR}/metadata"
LOCATION="us-central1"

print(f"PROJECT_ID: {PROJECT_ID}")
print(f"PROJECT_NBR: {PROJECT_NBR}")
print(f"LOCATION: {PROJECT_ID}")
print(f"DATA_SCAN_API_CREATE_ENDPOINT_PREFIX: {DATA_SCAN_API_CREATE_ENDPOINT_PREFIX}")
print(f"DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX: {DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX}")

## 1.0. [BigQuery] Create dataset and tables

### Create BigQuery datasets for the capstone

In [ ]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

CREATE SCHEMA `capstone_ds` OPTIONS (location = 'us-central1');
CREATE SCHEMA capstone_metadata_ds OPTIONS (location = 'us-central1');


### Create the metadata dataset and tables to persist Data Insights scan results

In [ ]:
%%bigquery --project {PROJECT_ID}

CREATE OR REPLACE TABLE `capstone_metadata_ds.dataset_description`
(
  dataset_description STRING
);

CREATE OR REPLACE TABLE `capstone_metadata_ds.dataset_table_relationships`
(
  table_1 STRING,
  table_1_column STRING,
  table_2 STRING,
  table_2_column STRING,
  join_type STRING
);

CREATE OR REPLACE TABLE `capstone_metadata_ds.table_column_descriptions`
(
  table_name STRING,
  column_name STRING,
  column_description STRING
);

CREATE OR REPLACE TABLE `capstone_metadata_ds.table_descriptions`
(
  name STRING,
  description STRING
);

### Create operational tables

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `capstone_ds.agent_activity_log`
(
  item_number STRING,
  omni_item_id STRING,
  activity_type STRING,
  activity_detail STRING,
  execution_date DATETIME,
  executed_by STRING
);

CREATE OR REPLACE TABLE `capstone_ds.procedure_error_log` (
  error_time DATETIME,
  procedure_name STRING,
  error_message STRING,
  statement_text STRING
);

### Create core tables

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `capstone_ds.customer_master`
 (customer_id STRING ,
  first_name STRING ,
  last_name STRING ,
  email STRING ,
  address STRING ,
  city STRING ,
  state_code STRING ,
  zip_code STRING ,
  country_code STRING ,
  phone_number STRING);

CREATE OR REPLACE TABLE `capstone_ds.demand_forecast_history`
(
  forecast_id STRING,
  forecast_run_time DATETIME,
  generated_by STRING,
  item_number STRING,
  item_name STRING,
  location_id STRING,
  omni_item_id STRING,
  forecast_timestamp TIMESTAMP,
  forecast_value FLOAT64,
  confidence_level FLOAT64,
  prediction_interval_lower_bound FLOAT64,
  prediction_interval_upper_bound FLOAT64,
  ai_forecast_status STRING

);

CREATE TABLE IF NOT EXISTS `capstone_ds.demand_forecast`
(
  forecast_id STRING,
  forecast_run_time DATETIME,
  generated_by STRING,
  item_number STRING,
  item_name STRING,
  location_id STRING,
  omni_item_id STRING,
  forecast_timestamp TIMESTAMP,
  forecast_value FLOAT64,
  confidence_level FLOAT64,
  prediction_interval_lower_bound FLOAT64,
  prediction_interval_upper_bound FLOAT64,
  ai_forecast_status STRING
);

CREATE OR REPLACE TABLE `capstone_ds.forecast_override_configs`
(
  item_number STRING,
  omni_item_id STRING,
  demand_surge_multiplier FLOAT64,
  demand_slump_multiplier FLOAT64,
  last_update_date DATE,
  last_updated_by STRING

);

CREATE OR REPLACE TABLE `capstone_ds.driver_master` (driver_id STRING ,    employee_id STRING );

CREATE OR REPLACE TABLE `capstone_ds.fleet_master` (vehicle_id STRING ,    license_plate STRING ,    vin STRING ,    model STRING ,    status STRING );

CREATE OR REPLACE TABLE `capstone_ds.forecast_activity_log`
(
  item_number STRING,
  omni_item_id STRING,
  forecast_id STRING,
  activity_type STRING,
  execution_date DATETIME,
  executed_by STRING
);

CREATE OR REPLACE TABLE `capstone_ds.demand_signal_log`
(
  item_number STRING,
  omni_item_id STRING,
  signal_indicator STRING, --SURGE / SLUMP
  comment STRING,
  signal_datetime DATETIME,
  notified_by STRING,
  is_active BOOL
);


CREATE OR REPLACE TABLE `capstone_ds.location_master` (location_id STRING ,    location_name STRING ,    location_type STRING ,    address STRING ,    city STRING ,    state_code STRING ,    zip_code STRING ,    country_code STRING ,    phone_number STRING );

CREATE OR REPLACE TABLE `capstone_ds.pos_transaction_items` (
  transaction_id STRING,
  item_number STRING,
  omni_item_id STRING,
  quantity INT,
  price DECIMAL,
  line_item_total DECIMAL);


CREATE OR REPLACE TABLE `capstone_ds.pos_transactions` (
  transaction_id STRING,
  location_id STRING,
  customer_id STRING,
  transaction_status STRING,
  transaction_date DATE,
  payment_type STRING,
  payment_total_dollar DECIMAL);


CREATE OR REPLACE TABLE `capstone_ds.product_docs_ref_data` (
  item_number STRING ,
  brand STRING,
  model_id STRING,
  pdf_name STRING,
  product_doc_type STRING,
  product_doc_gcs_uri STRING);

CREATE OR REPLACE TABLE `capstone_ds.product_master` (
  item_number STRING,
  appliance_type STRING,
  appliance_sub_type STRING,
  brand STRING,
  model_id STRING,
  omni_item_id STRING,
  description STRING,
  price NUMERIC,
  product_image_gcs_uri STRING,
  is_active STRING);

CREATE OR REPLACE TABLE `capstone_ds.product_suppliers`
(
  item_number STRING,
  omni_item_id STRING,
  supplier_id STRING,
  supplier_type STRING,
  lead_days_to_delivery INTEGER
);

---

CREATE OR REPLACE TABLE `capstone_ds.stock_allocation_plan` (
  allocation_date DATE ,
  item_number STRING,
  omni_item_id STRING,
  location_id STRING,
  quantity INTEGER,
  update_date DATE,
  is_current BOOL);

CREATE OR REPLACE TABLE capstone_ds.stock_master (
    stock_date DATE,
    item_number STRING,
    omni_item_id STRING,
    quantity_on_hand INT,
    reorder_point INT
);


CREATE OR REPLACE TABLE capstone_ds.stock_master_location (
    stock_date DATE,
    item_number STRING,
    omni_item_id STRING,
    location_id STRING,
    quantity_on_hand INT
);

CREATE OR REPLACE TABLE `capstone_ds.stock_movement` (
  movement_date DATE ,
  item_number STRING,
  omni_item_id STRING,
  movement_type STRING,
  location_id STRING,
  quantity_change INTEGER,
  reference_id STRING,
  reference_id_type STRING);

CREATE OR REPLACE TABLE `capstone_ds.stock_purchase_orders` (
  po_id STRING ,
  supplier_id STRING,
  order_date DATE,
  expected_date DATE,
  received_date DATE,
  total_cost DECIMAL,
  order_status  STRING,
  order_comment STRING);

CREATE OR REPLACE TABLE `capstone_ds.stock_purchase_order_items` (
  po_id STRING ,
  line_item_id STRING,
  item_number STRING,
  omni_item_id STRING,
  quantity_ordered  INTEGER,
  unit_price DECIMAL);

CREATE OR REPLACE TABLE capstone_ds.stock_thresholds (
item_number	STRING,
omni_item_id STRING,
stock_on_hand_across_stores	INT,
stock_at_each_store	INT,
total_stock_at_stores	INT,
total_stock_at_warehouse	INT,
average_sold_per_day_per_store	INT,
avg_sold_per_day_total	INT,
safety_stock_per_store	INT,
safety_stock_total	INT,
reorder_point_per_store	INT,
reorder_point_total	INT,
last_updated_date	DATE,
is_current STRING);

CREATE OR REPLACE TABLE capstone_ds.stock_transfer_orders (
    stock_transfer_order_id STRING,
    from_location_id STRING,
    to_location_id STRING,
    status STRING,
    vehicle_id STRING,
    reference_id STRING,
    reference_id_type STRING,
    transfer_type STRING,
    transfer_order_date DATE,
    fulfillment_date DATE
);

CREATE OR REPLACE TABLE capstone_ds.stock_transfer_order_items (
    stock_transfer_order_id STRING,
    item_number STRING,
    omni_item_id STRING,
    quantity INT
);

CREATE OR REPLACE TABLE `capstone_ds.supplier_master` (
  supplier_id STRING,
  supplier_name STRING,
  contact_name STRING,
  contact_email STRING,
  address STRING,
  city STRING,
  state_code STRING,
  zip_cd STRING,
  country_code STRING,
  phone_number STRING
);



## 2. [BigQuery] Create views

#### vw_supplier_fill_rate

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW `capstone_ds.vw_supplier_fill_rate` AS
WITH ordered_qty AS (
  -- Sum of quantities ordered per PO and Item
  SELECT
    po.po_id,
    po.supplier_id,
    po.order_date,
    poi.omni_item_id,
    SUM(poi.quantity_ordered) as total_ordered
  FROM `capstone_ds.stock_purchase_orders` po
  JOIN `capstone_ds.stock_purchase_order_items` poi ON po.po_id = poi.po_id
  GROUP BY 1, 2, 3, 4
),
received_qty AS (
  -- Sum of quantities actually received (matched by reference_id/PO_id)
  SELECT
    reference_id as po_id,
    omni_item_id,
    SUM(quantity_change) as total_received
  FROM `capstone_ds.stock_movement`
  WHERE movement_type = 'SUPPLIER_DELIVERY_TO_WAREHOUSE'
  GROUP BY 1, 2
)

SELECT
  o.supplier_id,
  o.order_date,
  o.po_id,
  o.omni_item_id,
  o.total_ordered,
  COALESCE(r.total_received, 0) as total_received,
  -- Calculate Fill Rate %
  SAFE_DIVIDE(COALESCE(r.total_received, 0), o.total_ordered) * 100 as fill_rate_percentage,
  -- Identify the status
  CASE
    WHEN COALESCE(r.total_received, 0) = 0 THEN 'NON_DELIVERY'
    WHEN COALESCE(r.total_received, 0) < o.total_ordered THEN 'SHORT_SHIP'
    WHEN COALESCE(r.total_received, 0) = o.total_ordered THEN 'FULL_DELIVERY'
    ELSE 'OVER_SHIP'
  END as fulfillment_status
FROM ordered_qty o
LEFT JOIN received_qty r ON o.po_id = r.po_id AND o.omni_item_id = r.omni_item_id;


#### vw_weighted_lead_time

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
create or replace view `capstone_ds.vw_weighted_lead_time` AS
WITH lead_time_data AS (
  SELECT
    po.supplier_id,
    poi.omni_item_id,
    po.po_id,
    -- Calculate days between order and arrival
    DATE_DIFF(sm.movement_date, po.order_date, DAY) as actual_lead_time_days,
    sm.quantity_change as qty_received
  FROM `capstone_ds.stock_purchase_orders` po
  JOIN `capstone_ds.stock_purchase_order_items` poi ON po.po_id = poi.po_id
  JOIN `capstone_ds.stock_movement` sm ON po.po_id = sm.reference_id
    AND poi.omni_item_id = sm.omni_item_id
  WHERE sm.movement_type = 'SUPPLIER_DELIVERY_TO_WAREHOUSE'
    AND po.order_status = 'FULFILLED'
),

actual_metrics AS (
  SELECT
    supplier_id,
    omni_item_id,
    COUNT(DISTINCT po_id) as total_pos_analyzed,
    SUM(qty_received) as total_units_received,
    -- Weighted Lead Time Calculation
    SAFE_DIVIDE(SUM(actual_lead_time_days * qty_received), SUM(qty_received)) as weighted_actual_lead_time
  FROM lead_time_data
  GROUP BY 1, 2
),

contract_data AS (
  SELECT
    supplier_id,
    omni_item_id,
    item_number,
    lead_days_to_delivery as contracted_lead_time
  FROM `capstone_ds.product_suppliers`
)

SELECT
  c.item_number,
  c.omni_item_id,
  c.supplier_id,
  c.contracted_lead_time,
  ROUND(a.weighted_actual_lead_time, 2) as actual_weighted_lead_time,
  -- Performance Variance (Positive = Supplier is Late)
  ROUND(a.weighted_actual_lead_time - c.contracted_lead_time, 2) as lead_time_variance,
  -- Reliability Index
  CASE
    WHEN a.weighted_actual_lead_time <= c.contracted_lead_time THEN 'ON_TIME_OR_EARLY'
    WHEN (a.weighted_actual_lead_time - c.contracted_lead_time) <= 2 THEN 'MARGINAL_DELAY'
    ELSE 'CRITICAL_DELAY'
  END as performance_tier
FROM contract_data c
JOIN actual_metrics a ON c.supplier_id = a.supplier_id AND c.omni_item_id = a.omni_item_id
ORDER BY lead_time_variance DESC;

#### vw_demand_signal_log

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW `capstone_ds.vw_demand_signal_log` AS
select * from capstone_ds.demand_signal_log
order by signal_datetime desc;


#### vw_sales_history

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW
    `capstone_ds.vw_sales_history` AS
SELECT
    t.transaction_id,
    t.location_id,
    t.customer_id,
    t.transaction_status,
    t.transaction_date,
    t.payment_type,
    t.payment_total_dollar,
    ti.item_number,
    ti.omni_item_id,
    ti.quantity,
    pm.description,
    pm.appliance_sub_type,
    pm.brand,
    ti.price,
    ti.line_item_total,
    cm.city
  FROM
    `capstone_ds.pos_transactions` AS t
    INNER JOIN `capstone_ds.pos_transaction_items` AS ti ON t.transaction_id = ti.transaction_id
    INNER JOIN `capstone_ds.product_master` AS pm ON ti.omni_item_id = pm.omni_item_id
    INNER JOIN `capstone_ds.customer_master` AS cm ON t.customer_id = cm.customer_id

#### vw_aggr_sales_by_item

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

-- SALES AGGREGATED BY DATE, STORE, ITEM_NM
CREATE OR REPLACE VIEW
  `capstone_ds.vw_aggr_sales_by_item` AS (
WITH top_sellers AS(
      SELECT
        item_number,
        omni_item_id,
        description as item_nm,
        SUM(quantity) AS total_quantity
    FROM
        `capstone_ds.vw_sales_history`
    GROUP BY
        item_number,
        omni_item_id,
        description
    ORDER BY total_quantity DESC
    LIMIT 25
)
SELECT
    transaction_date,
    location_id,
    item_number,
    omni_item_id,
    description as item_nm,
     SUM(quantity) AS total_quantity
FROM
    `capstone_ds.vw_sales_history`
GROUP BY transaction_date, location_id,
    item_number,
    omni_item_id,
    description
HAVING
    transaction_date BETWEEN '2025-01-01' AND '2026-01-31'
    AND omni_item_id IN (SELECT omni_item_id FROM top_sellers)
);

#### vw_stock_reconciliation

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE view capstone_ds.vw_stock_reconciliation AS

WITH
-- 1. Master list of products to ensure no gaps
products AS (
  SELECT DISTINCT item_number, omni_item_id
  FROM `capstone_ds.product_master`
  WHERE is_active='Y'
),

-- 2. Stock received from supplier orders
received AS (
  SELECT omni_item_id, item_number, SUM(quantity_change) as total_received
  FROM `capstone_ds.stock_movement`
  WHERE movement_type ='SUPPLIER_DELIVERY_TO_WAREHOUSE'
  GROUP BY 1, 2
),

-- 3. Stock sales from POS system
pos_sales AS (
  SELECT omni_item_id, SUM(quantity) AS pos_total_sold
  FROM `capstone_ds.pos_transaction_items`
  GROUP BY 1
),

-- 4. Stock movement log for POS transactions (internal ledger)
log_sales AS (
  SELECT item_number, omni_item_id, SUM(quantity_change) as log_total_sold
  FROM `capstone_ds.stock_movement`
  WHERE movement_type ='SALE'
  GROUP BY 1, 2
),

-- 5. Stock on hand from stock_master (most recent date)
on_hand AS (
  SELECT item_number, omni_item_id, SUM(quantity_on_hand) as current_stock
  FROM `capstone_ds.stock_master`
  WHERE stock_date = (SELECT MAX(stock_date) FROM `capstone_dsock_master`)
  GROUP BY 1, 2
)

-- Final Join to bring it all together
SELECT
  p.item_number,
  p.omni_item_id,
  COALESCE(r.total_received, 0) as received_qty,
  COALESCE(ps.pos_total_sold, 0) as pos_sold_qty,
  COALESCE(ls.log_total_sold, 0) as movement_log_sold_qty,
  COALESCE(oh.current_stock, 0) as stock_on_hand,
  -- Reconciliation logic: Difference between POS and the Internal Movement Log
  (COALESCE(ps.pos_total_sold, 0) + COALESCE(ls.log_total_sold, 0)) as sales_variance,
  (COALESCE(r.total_received, 0) - (COALESCE(ps.pos_total_sold, 0)  + COALESCE(oh.current_stock, 0))) as on_hand_variance
FROM products p
LEFT JOIN received r  ON p.omni_item_id = r.omni_item_id
LEFT JOIN pos_sales ps ON p.omni_item_id = ps.omni_item_id
LEFT JOIN log_sales ls ON p.omni_item_id = ls.omni_item_id
LEFT JOIN on_hand   oh ON p.omni_item_id = oh.omni_item_id
ORDER BY p.item_number;

#### vw_stock_movement_expanded

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE view capstone_ds.vw_stock_movement_expanded AS
select 'INBOUND_TO_STORES' as direction,movement_date,item_number, omni_item_id,location_id,quantity_change,movement_type,reference_id_type
from capstone_ds.stock_movement where movement_type in ('STOCK_REPLESHIMENT_FROM_WAREHOUSE','INITIAL_STOCK_FROM_WAREHOUSE')
union ALL
select 'OUTBOUND_FROM_STORES' as direction,movement_date,item_number, omni_item_id,location_id,quantity_change,movement_type,reference_id_type
from capstone_ds.stock_movement where movement_type in ('SALE')
union ALL
select 'INBOUND_TO_WAREHOUSE' as direction,movement_date,item_number, omni_item_id,location_id,quantity_change,movement_type,reference_id_type
from capstone_ds.stock_movement where movement_type in ('SUPPLIER_DELIVERY_TO_WAREHOUSE')
union ALL
select 'OUTBOUND_FROM_WAREHOUSE' as direction,movement_date,item_number, omni_item_id,location_id,quantity_change,movement_type,reference_id_type
from capstone_ds.stock_movement where movement_type in ('STOCK_REPLESHIMENT_TO_STORE','WAREHOUSE_TO_STORE')
ORDER BY direction,location_id, movement_date;


#### vw_stock_movement_summary

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE view capstone_ds.vw_stock_movement_summary AS
WITH movement_summary AS (
    SELECT 'SUPPLIER_DELIVERY' as direction, item_number, omni_item_id, sum(quantity_change) as qty
    FROM capstone_ds.stock_movement
    WHERE movement_type IN ('STOCK_REPLESHIMENT_FROM_WAREHOUSE','INITIAL_STOCK_FROM_WAREHOUSE')
    GROUP BY 1, 2, 3

    UNION ALL
    SELECT 'WAREHOUSE_DISTRIBUTION', item_number, omni_item_id, -1* sum(quantity_change)
    FROM capstone_ds.stock_movement
    WHERE movement_type IN ('SALE')
    GROUP BY 1, 2, 3

    UNION ALL
    SELECT 'STORE_REPLENISHMENT', item_number, omni_item_id, sum(quantity_change)
    FROM capstone_ds.stock_movement
    WHERE movement_type IN ('SUPPLIER_DELIVERY_TO_WAREHOUSE')
    GROUP BY 1, 2, 3

    UNION ALL
    SELECT 'SALES', item_number, omni_item_id, -1* sum(quantity_change)
    FROM capstone_ds.stock_movement
    WHERE movement_type IN ('STOCK_REPLESHIMENT_TO_STORE','WAREHOUSE_TO_STORE')
    GROUP BY 1, 2, 3
)

SELECT * FROM movement_summary
PIVOT(
    SUM(qty)
    FOR direction IN (
        'SUPPLIER_DELIVERY',
        'WAREHOUSE_DISTRIBUTION',
        'STORE_REPLENISHMENT',
        'SALES'

    )
)
ORDER BY item_number, omni_item_id;

#### vw_stock_reorder_points

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE view capstone_ds.vw_stock_reorder_points AS
WITH daily_sales AS (
  SELECT
    pt.transaction_date,
    pti.item_number,
    pti.omni_item_id,
    SUM(pti.quantity) AS daily_sold
  FROM `capstone_ds.pos_transaction_items` pti
  JOIN `capstone_ds.pos_transactions` pt ON pt.transaction_id = pti.transaction_id
  GROUP BY 1, 2, 3
),
running_totals AS (
  SELECT
    *,
    SUM(daily_sold) OVER (
      PARTITION BY omni_item_id
      ORDER BY transaction_date
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_quantity_sold
  FROM daily_sales
),
bin_calculations AS (
  SELECT
    rt.*,
    ps.supplier_id,
    -- Trigger reorder 10 units before every multiple of 180
    FLOOR((running_quantity_sold + 10) / 180) AS current_reorder_bin,
    LAG(FLOOR((running_quantity_sold + 10) / 180)) OVER (
      PARTITION BY rt.omni_item_id ORDER BY transaction_date
    ) AS prev_reorder_bin
  FROM running_totals rt
  LEFT JOIN `capstone_ds.product_suppliers` ps
    ON rt.item_number = ps.item_number AND rt.omni_item_id = ps.omni_item_id
)

SELECT
  transaction_date,
  item_number,
  omni_item_id,
  daily_sold,
  running_quantity_sold,
  supplier_id,
  CASE
    WHEN current_reorder_bin > COALESCE(prev_reorder_bin, 0) THEN TRUE
    ELSE FALSE
  END AS reorder_flag
FROM bin_calculations;

#### vw_inventory_aging

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_inventory_aging as
WITH LastSale AS
 (SELECT pti.item_number, pti.omni_item_id, MAX(pt.transaction_date) AS last_sale_date
  FROM `capstone_ds.pos_transaction_items` AS pti
  JOIN `capstone_ds.pos_transactions` AS pt
  ON pti.transaction_id = pt.transaction_id GROUP BY pti.item_number,pti.omni_item_id ),
CurrentStock AS
 (SELECT item_number, omni_item_id, quantity_on_hand FROM (SELECT item_number, omni_item_id, quantity_on_hand, ROW_NUMBER() OVER (PARTITION BY item_number,omni_item_id ORDER BY stock_date DESC) as rn
  FROM `capstone_ds.stock_master` ) WHERE rn = 1 )
 SELECT cs.item_number, cs.omni_item_id, pm.description, cs.quantity_on_hand, ls.last_sale_date, DATE_DIFF(CURRENT_DATE(), ls.last_sale_date, DAY) AS days_since_last_sale
 FROM CurrentStock AS cs
 JOIN LastSale AS ls ON cs.omni_item_id = ls.omni_item_id
 JOIN `capstone_ds.product_master` AS pm ON cs.omni_item_id = pm.omni_item_id and cs.item_number=pm.item_number
 WHERE cs.quantity_on_hand > 0 ORDER BY days_since_last_sale DESC;


#### vw_stock_master_location

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
CREATE OR REPLACE VIEW capstone_ds.vw_stock_master_location as
select * from capstone_ds.stock_master_location
order by omni_item_id, location_id, stock_date;

#### vw_stock_master

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
CREATE OR REPLACE VIEW capstone_ds.vw_stock_master as
select * from capstone_ds.stock_master
order by omni_item_id,stock_date;

#### vw_stock_movement

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW capstone_ds.vw_stock_movement as
select * from capstone_ds.stock_movement order by movement_date,omni_item_id,location_id,movement_type;

#### vw_stock_purchase_orders

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW capstone_ds.vw_stock_purchase_orders AS
select po.po_id,po.supplier_id,po.order_date,po.received_date, po.total_cost, po.order_status,po.order_comment,poi.item_number,poi.omni_item_id,poi.quantity_ordered,poi.unit_price
from capstone_ds.stock_purchase_orders po join capstone_ds.stock_purchase_order_items poi on (po.po_id=poi.po_id)
order by order_date, supplier_id;

#### vw_stock_transfer_orders

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW capstone_ds.vw_stock_transfer_orders as
select sto.stock_transfer_order_id,sto.transfer_order_date,sto.fulfillment_date,sto.from_location_id,sto.to_location_id, sto.reference_id_type,stoi.item_number,stoi.omni_item_id,stoi.quantity
from capstone_ds.stock_transfer_orders sto
join capstone_ds.stock_transfer_order_items stoi
on (sto.stock_transfer_order_id=stoi.stock_transfer_order_id)
order by stoi.omni_item_id,sto.transfer_order_date,sto.from_location_id,sto.to_location_id;

#### vw_stock_transfer_aggr

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW capstone_ds.vw_stock_transfer_aggr as
select sto.transfer_order_date,sto.from_location_id,'STORES' as to_location_id, sto.reference_id_type,stoi.item_number,stoi.omni_item_id,sum(stoi.quantity) as quantity
from capstone_ds.stock_transfer_orders sto
join capstone_ds.stock_transfer_order_items stoi
on (sto.stock_transfer_order_id=stoi.stock_transfer_order_id)
group by sto.transfer_order_date,sto.from_location_id,sto.reference_id_type,stoi.item_number,stoi.omni_item_id
order by stoi.omni_item_id,sto.transfer_order_date,sto.from_location_id;



#### vw_stock_reconciliation

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE view capstone_ds.vw_stock_reconciliation AS

WITH
-- 1. Master list of products to ensure no gaps
products AS (
  SELECT DISTINCT item_number, omni_item_id
  FROM `capstone_ds.product_master`
  WHERE is_active='Y'
),

-- 2. Stock received from supplier orders
received AS (
  SELECT omni_item_id, item_number, SUM(quantity_change) as total_received
  FROM `capstone_ds.stock_movement`
  WHERE movement_type ='SUPPLIER_DELIVERY_TO_WAREHOUSE'
  GROUP BY 1, 2
),

-- 3. Stock sales from POS system
pos_sales AS (
  SELECT omni_item_id, SUM(quantity) AS pos_total_sold
  FROM `capstone_ds.pos_transaction_items`
  GROUP BY 1
),

-- 4. Stock movement log for POS transactions (internal ledger)
log_sales AS (
  SELECT item_number, omni_item_id, SUM(quantity_change) as log_total_sold
  FROM `capstone_ds.stock_movement`
  WHERE movement_type ='SALE'
  GROUP BY 1, 2
),

-- 5. Stock on hand from stock_master (most recent date)
on_hand AS (
  SELECT item_number, omni_item_id, SUM(quantity_on_hand) as current_stock
  FROM `capstone_ds.stock_master`
  WHERE stock_date = (SELECT MAX(stock_date) FROM `capstone_ds.stock_master`)
  GROUP BY 1, 2
)

-- Final Join to bring it all together
SELECT
  p.item_number,
  p.omni_item_id,
  COALESCE(r.total_received, 0) as received_qty,
  COALESCE(ps.pos_total_sold, 0) as pos_sold_qty,
  COALESCE(ls.log_total_sold, 0) as movement_log_sold_qty,
  COALESCE(oh.current_stock, 0) as stock_on_hand,
  -- Reconciliation logic: Difference between POS and the Internal Movement Log
  (COALESCE(ps.pos_total_sold, 0) + COALESCE(ls.log_total_sold, 0)) as sales_variance,
  (COALESCE(r.total_received, 0) - (COALESCE(ps.pos_total_sold, 0)  + COALESCE(oh.current_stock, 0))) as on_hand_variance
FROM products p
LEFT JOIN received r  ON p.omni_item_id = r.omni_item_id
LEFT JOIN pos_sales ps ON p.omni_item_id = ps.omni_item_id
LEFT JOIN log_sales ls ON p.omni_item_id = ls.omni_item_id
LEFT JOIN on_hand   oh ON p.omni_item_id = oh.omni_item_id
ORDER BY p.item_number;

#### vw_suggested_reorder

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_suggested_reorder as
WITH velocity_data AS (
  -- Calculate average daily sales over the last 30 days
  SELECT
    pti.omni_item_id,
    SUM(pti.quantity) / 30 as daily_velocity
  FROM `capstone_ds.pos_transaction_items` pti
  JOIN `capstone_ds.pos_transactions` pt ON pti.transaction_id = pt.transaction_id
  WHERE pt.transaction_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  GROUP BY 1
),

lead_time_data AS (
  -- Use the weighted lead time per item/supplier
  SELECT
    omni_item_id,
    supplier_id,
    lead_days_to_delivery as contracted_lead_time
  FROM `capstone_ds.product_suppliers`
),

inventory_status AS (
  -- Get current stock on hand
  SELECT
    omni_item_id,
    SUM(quantity_on_hand) as current_stock
  FROM `capstone_ds.stock_master`
  WHERE stock_date = (SELECT MAX(stock_date) FROM `capstone_ds.stock_master`)
  GROUP BY 1
)

SELECT
  p.item_number,
  p.omni_item_id,
  l.supplier_id,
  ROUND(v.daily_velocity, 2) as daily_velocity,
  l.contracted_lead_time,
  i.current_stock,
  -- Reorder Point = (Lead Time * Velocity) + Safety Buffer (e.g., 2 days of sales)
  ROUND((l.contracted_lead_time * v.daily_velocity) + (v.daily_velocity * 2), 0) as reorder_point,

  -- Logic: If current stock is below reorder point, suggest an order
  CASE
    WHEN i.current_stock <= ROUND((l.contracted_lead_time * v.daily_velocity) + (v.daily_velocity * 2), 0) THEN 'YES'
    ELSE 'NO'
  END as reorder_required,

  -- Suggestion based on your "Multiple of 180" business rule
  CASE
    WHEN i.current_stock <= ROUND((l.contracted_lead_time * v.daily_velocity) + (v.daily_velocity * 2), 0)
    THEN 180 -- Standard replenishment unit
    ELSE 0
  END as suggested_order_qty

FROM `capstone_ds.product_master` p
JOIN velocity_data v ON p.omni_item_id = v.omni_item_id
JOIN lead_time_data l ON p.omni_item_id = l.omni_item_id
JOIN inventory_status i ON p.omni_item_id = i.omni_item_id
WHERE i.current_stock <= (l.contracted_lead_time * v.daily_velocity) + (v.daily_velocity * 2)
ORDER BY daily_velocity DESC

### vw_avg_rate_of_sale

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW capstone_ds.vw_avg_rate_of_sale AS
WITH daily_item_sales AS (
  -- Aggregate sales to the Item / Location / Date level
  -- Uses POS transaction items joined with main transaction headers for dates
  SELECT
    p.item_number,
    p.omni_item_id,
    t.location_id,
    t.transaction_date,
    SUM(p.quantity) AS daily_units
  FROM
    `capstone_ds.pos_transaction_items` p
  JOIN
    `capstone_ds.pos_transactions` t ON p.transaction_id = t.transaction_id
  WHERE
    -- Typically planners look at a 28-day window for stability
    t.transaction_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 28 DAY)
  GROUP BY 1, 2, 3, 4
),

ros_summary AS (
  -- Calculate ROS: Total Units / Total Store-Days where sales occurred
  SELECT
    item_number,
    omni_item_id,
    SUM(daily_units) AS total_units_period,
    COUNT(DISTINCT location_id) AS active_store_count,
    SAFE_DIVIDE(SUM(daily_units), COUNT(DISTINCT CONCAT(location_id, CAST(transaction_date AS STRING)))) AS rate_of_sale
  FROM
    daily_item_sales
  GROUP BY 1, 2
)

SELECT
  m.item_number,
  m.omni_item_id,
  m.brand,
  m.appliance_type,
  r.rate_of_sale,
  r.total_units_period,
  r.active_store_count,
  -- Segment based on velocity thresholds
  CASE
    WHEN r.rate_of_sale > 0.5 THEN 'High Velocity'
    WHEN r.rate_of_sale BETWEEN 0.1 AND 0.5 THEN 'Medium Velocity'
    ELSE 'Slow Mover'
  END AS velocity_segment
FROM
  ros_summary r
JOIN
  `capstone_ds.product_master` m ON r.item_number = m.item_number
ORDER BY
  r.rate_of_sale DESC;

### vw_weeks_of_supply

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_weeks_of_supply AS
WITH ros_data AS (
  -- Reuse the ROS calculation logic per location
  SELECT
    p.item_number,
    p.omni_item_id,
    t.location_id,
    -- Weekly ROS (Units per store per day * 7 days)
    SAFE_DIVIDE(SUM(p.quantity), 28) * 7 AS weekly_ros
  FROM
    `capstone_ds.pos_transaction_items` p
  JOIN
    `capstone_ds.pos_transactions` t ON p.transaction_id = t.transaction_id
  WHERE
    t.transaction_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 28 DAY)
  GROUP BY 1, 2, 3
),

inventory_on_hand AS (
  -- Get the most recent stock levels per location
  SELECT
    item_number,
    omni_item_id,
    location_id,
    quantity_on_hand
  FROM
    `capstone_ds.stock_master_location`
  WHERE
    stock_date = (SELECT MAX(stock_date) FROM `capstone_ds.stock_master_location`)
)

SELECT
  i.location_id,
  l.location_name,
  i.item_number,
  i.omni_item_id,
  i.quantity_on_hand,
  r.weekly_ros,
  -- WOS = Current Stock / Units Sold per Week
  SAFE_DIVIDE(i.quantity_on_hand, r.weekly_ros) AS weeks_of_supply,
  -- Flag items that need immediate reorder based on WOS
  CASE
    WHEN SAFE_DIVIDE(i.quantity_on_hand, r.weekly_ros) < 2 THEN 'CRITICAL: Stockout Risk'
    WHEN SAFE_DIVIDE(i.quantity_on_hand, r.weekly_ros) > 12 THEN 'EXCESS: Aged Stock'
    ELSE 'Healthy'
  END AS inventory_status
FROM
  inventory_on_hand i
LEFT JOIN
  ros_data r ON i.item_number = r.item_number
             AND i.omni_item_id = r.omni_item_id
             AND i.location_id = r.location_id
JOIN
  `capstone_ds.location_master` l ON i.location_id = l.location_id
ORDER BY
  weeks_of_supply ASC;

### vw_stock_transfer_fulfill

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE VIEW capstone_ds.vw_stock_transfer_fulfill AS
SELECT
    t1.stock_transfer_order_id,
    t1.transfer_order_date,
    t1.fulfillment_date,
    t1.from_location_id,
    t1.to_location_id,
    t1.status,
    t2.item_number,
    t2.omni_item_id,
    t2.quantity
FROM
    `capstone_ds.stock_transfer_orders` AS t1
JOIN
    `capstone_ds.stock_transfer_order_items` AS t2
    ON t1.stock_transfer_order_id = t2.stock_transfer_order_id
ORDER BY
    t1.transfer_order_date DESC;

### vw_transfer_volume

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--Location Based Transfer Volume Report

create or replace view capstone_ds.vw_transfer_volume AS
SELECT
    from_location_id,
    to_location_id,
    SUM(quantity) AS total_units_transferred,
    COUNT(*) AS total_transfer_orders
FROM
    `capstone_ds.vw_stock_transfer_aggr`
GROUP BY
    from_location_id, to_location_id
ORDER BY
    total_units_transferred DESC;

### vw_fleet_usage

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_fleet_usage AS
SELECT
    f.vehicle_id,
    t.stock_transfer_order_id,
    t.transfer_order_date,
    t.fulfillment_date,
    t.to_location_id,
    t.status AS transfer_status
FROM
    `capstone_ds.fleet_master` f
JOIN
    `capstone_ds.stock_transfer_orders` t ON f.vehicle_id = t.vehicle_id
WHERE
    t.status = 'DELIVERED'
ORDER BY
    t.transfer_order_date DESC;


### vw_fleet_items

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_fleet_items AS
SELECT
    t.vehicle_id,
    COUNT(t.stock_transfer_order_id) AS total_transfers,
    SUM(ti.quantity) AS total_items_moved
FROM
    `capstone_ds.stock_transfer_orders` t
JOIN
    `capstone_ds.fleet_master` f ON t.vehicle_id = f.vehicle_id
JOIN
    `capstone_ds.stock_transfer_order_items` ti ON t.stock_transfer_order_id = ti.stock_transfer_order_id
WHERE
    t.status = 'DELIVERED'
GROUP BY
    1
ORDER BY
    total_transfers DESC;

### vw_on_hand_stock

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_on_hand_stock AS
WITH RankedInventory AS (SELECT omni_item_id, quantity_on_hand, reorder_point, stock_date, ROW_NUMBER() OVER (PARTITION BY omni_item_id ORDER BY stock_date DESC) as rn FROM `capstone_ds.stock_master` ) SELECT omni_item_id, quantity_on_hand, reorder_point, stock_date AS latest_stock_date FROM RankedInventory WHERE rn = 1 ORDER BY omni_item_id

### vw_on_hand_loc_stock

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

create or replace view capstone_ds.vw_on_hand_loc_stock
WITH RankedInventory AS (
  SELECT
    item_number,
    omni_item_id,
    location_id,
    quantity_on_hand,
    stock_date,
    -- Get the most recent record for each item at each specific location
    ROW_NUMBER() OVER (PARTITION BY omni_item_id, location_id ORDER BY stock_date DESC) as rn
  FROM `capstone_ds.stock_master_location`
)
SELECT
    ri.item_number,
    ri.omni_item_id,
    l.location_name,
    ri.quantity_on_hand,
    -- Reorder point is typically managed at the item level in stock_master
    sm.reorder_point
FROM RankedInventory ri
JOIN `capstone_ds.location_master` l ON ri.location_id = l.location_id
JOIN `capstone_ds.stock_master` sm ON ri.omni_item_id = sm.omni_item_id AND ri.stock_date = sm.stock_date
WHERE ri.rn = 1
  AND ri.quantity_on_hand < sm.reorder_point
ORDER BY l.location_name, ri.omni_item_id;

## 3. [BigQuery] Load tables

### Create Load Utils

In [ ]:
def run_bigquery_sql(sql):
  import time
  from google.cloud import bigquery
  client = bigquery.Client(location=f"{LOCATION}")

  if (sql.startswith("SELECT") or sql.startswith("WITH")):
      df_result = client.query(sql).to_dataframe()
      return df_result
  else:
    job_config = bigquery.QueryJobConfig(priority=bigquery.QueryPriority.INTERACTIVE)
    query_job = client.query(sql, job_config=job_config)

    # Check on the progress by getting the job's updated state.
    query_job = client.get_job(
        query_job.job_id, location=query_job.location
    )
    print("Job {} is currently in state {} with error result of {}".format(query_job.job_id, query_job.state, query_job.error_result))

    while query_job.state != "DONE":
      time.sleep(2)
      query_job = client.get_job(
          query_job.job_id, location=query_job.location
          )
      print("Job {} is currently in state {} with error result of {}".format(query_job.job_id, query_job.state, query_job.error_result))

    if query_job.error_result == None:
      return True
    else:
      raise Exception(query_job.error_result)

### Product Master

In [ ]:
LOAD_PRODUCT_MASTER_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.product_master(
  item_number STRING,
  appliance_type STRING,
  appliance_sub_type STRING,
  brand STRING,
  model_id STRING,
  omni_item_id STRING,
  description STRING,
  price NUMERIC,
  product_image_gcs_uri STRING,
  is_active STRING
)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/product_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_PRODUCT_MASTER_SQL)

In [ ]:
UPDATE_PRODUCT_MASTER_SQL = f"""
UPDATE
  capstone_ds.product_master
SET
  -- Set the column to its new value
  product_image_gcs_uri = REGEXP_REPLACE(REPLACE(REPLACE(REPLACE(product_image_gcs_uri, 'PROJECT_NBR', \"{PROJECT_NBR}\"),'rscw-workshop-fridge-stage-','capstone_stage_'),'unstructured-data/',''), r'[{{}}]', ''),
  brand=UPPER(brand)
-- Optional: A WHERE clause can be added to filter which rows are affected
WHERE 1=1;
"""

run_bigquery_sql(UPDATE_PRODUCT_MASTER_SQL)



### Product Docs

In [ ]:
LOAD_PRODUCT_DOCS_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.product_docs_ref_data
(item_number STRING ,
  brand STRING,
  model_id STRING,
  pdf_name STRING,
  product_doc_type STRING,
  product_doc_gcs_uri STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/product_docs_ref_data/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_PRODUCT_DOCS_SQL)

In [ ]:
UPDATE_PRODUCT_DOCS_SQL = f"""
UPDATE
  capstone_ds.product_docs_ref_data
SET
  product_doc_gcs_uri = REPLACE(REPLACE(product_doc_gcs_uri, 'gs://rscw-workshop-fridge-stage-606804615020/unstructured-data/', 'gs://capstone_stage_{PROJECT_NBR}/'),'user-guides/refrigerators','fridge_userguides'),
  brand=UPPER(brand)
WHERE 1=1;
"""

run_bigquery_sql(UPDATE_PRODUCT_DOCS_SQL)

### Demand Forecast - Override Configs

In [ ]:
LOAD_FORECAST_OVERRIDES_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.forecast_override_configs
(item_number STRING,
  omni_item_id STRING,
  demand_surge_multiplier FLOAT64,
  demand_slump_multiplier FLOAT64,
  last_update_date DATE,
  last_updated_by STRING)

FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/forecast_override_configs/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_FORECAST_OVERRIDES_SQL)

### Supplier and Product Suppliers

In [ ]:
LOAD_SUPPLIER_MASTER_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.supplier_master
(supplier_id STRING,
  supplier_name STRING,
  contact_name STRING,
  contact_email STRING,
  address STRING,
  city STRING,
  state_code STRING,
  zip_cd STRING,
  country_code STRING,
  phone_number STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/supplier_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_SUPPLIER_MASTER_SQL)

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

delete from capstone_ds.product_suppliers
where 1=1;

insert into capstone_ds.product_suppliers(item_number, omni_item_id,supplier_id,supplier_type,lead_days_to_delivery)
select item_number,omni_item_id,UPPER(brand),'primary',7
from capstone_ds.product_master;

update capstone_ds.product_suppliers
set supplier_id=trim(supplier_id) where 1=1;

update capstone_ds.supplier_master
set supplier_id=trim(supplier_id) where 1=1;

### Customer Master

In [ ]:
LOAD_CUSTOMER_MASTER_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.customer_master
(customer_id STRING ,
  first_name STRING ,
  last_name STRING ,
  email STRING ,
  address STRING ,
  city STRING ,
  state_code STRING ,
  zip_code STRING ,
  country_code STRING ,
  phone_number STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/customer_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_CUSTOMER_MASTER_SQL)

### Driver, Fleet, Location

In [ ]:
LOAD_DRIVER_MASTER_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.driver_master
(driver_id STRING ,    employee_id STRING )
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/driver_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""


run_bigquery_sql(LOAD_DRIVER_MASTER_SQL)

In [ ]:
LOAD_FLEET_MASTER_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.fleet_master(vehicle_id STRING ,    license_plate STRING ,    vin STRING ,    model STRING ,    status STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/fleet_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""


run_bigquery_sql(LOAD_FLEET_MASTER_SQL)

In [ ]:
LOAD_LOCATION_MASTER_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.location_master(location_id STRING ,    location_name STRING ,    location_type STRING ,    address STRING ,    city STRING ,    state_code STRING ,    zip_code STRING ,    country_code STRING ,    phone_number STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/location_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_LOCATION_MASTER_SQL)


### POS Transaction Items and POS Transactions

In [ ]:
LOAD_pos_transactions_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.pos_transactions(
  transaction_id STRING,
  location_id STRING,
  customer_id STRING,
  transaction_status STRING,
  transaction_date DATE,
  payment_type STRING,
  payment_total_dollar DECIMAL)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/pos_transactions/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_pos_transactions_SQL)

In [ ]:
LOAD_pos_transaction_items_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.pos_transaction_items(
  transaction_id STRING,
  item_number STRING,
  omni_item_id STRING,
  quantity INT,
  price DECIMAL,
  line_item_total DECIMAL
)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/pos_transaction_items/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_pos_transaction_items_SQL)

### Stock Master and Stock Master Location

In [ ]:
LOAD_stock_master_location_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_master_location(stock_date DATE,
    item_number STRING,
    omni_item_id STRING,
    location_id STRING,
    quantity_on_hand INT)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_master_location/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_stock_master_location_SQL)

In [ ]:
LOAD_stock_master_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_master(stock_date DATE,
    item_number STRING,
    omni_item_id STRING,
    quantity_on_hand INT,
    reorder_point INT)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_master/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_stock_master_SQL)

### Stock Movement

In [ ]:
LOAD_stock_movement_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_movement(movement_date DATE ,
  item_number STRING,
  omni_item_id STRING,
  movement_type STRING,
  location_id STRING,
  quantity_change INTEGER,
  reference_id STRING,
  reference_id_type STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_movement/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_stock_movement_SQL)

### Stock Purchase Orders & Stock Purchase Order Items

In [ ]:
LOAD_stock_purchase_orders_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_purchase_orders(po_id STRING ,
  supplier_id STRING,
  order_date DATE,
  expected_date DATE,
  received_date DATE,
  total_cost DECIMAL,
  order_status  STRING,
  order_comment STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_purchase_orders/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_stock_purchase_orders_SQL)

In [ ]:
LOAD_stock_purchase_order_items_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_purchase_order_items(po_id STRING ,
  line_item_id STRING,
  item_number STRING,
  omni_item_id STRING,
  quantity_ordered  INTEGER,
  unit_price DECIMAL)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_purchase_order_items/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_stock_purchase_order_items_SQL)

In [ ]:
run_bigquery_sql("DELETE FROM capstone_ds.stock_purchase_orders WHERE order_status='PENDING'")

In [ ]:
run_bigquery_sql("DELETE FROM capstone_ds.stock_purchase_order_items WHERE po_id NOT IN (SELECT po_id FROM capstone_ds.stock_purchase_orders )")

### Stock thresholds

In [ ]:
LOAD_stock_thresholds_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_thresholds(item_number	STRING,
omni_item_id STRING,
stock_on_hand_across_stores	INT,
stock_at_each_store	INT,
total_stock_at_stores	INT,
total_stock_at_warehouse	INT,
average_sold_per_day_per_store	INT,
avg_sold_per_day_total	INT,
safety_stock_per_store	INT,
safety_stock_total	INT,
reorder_point_per_store	INT,
reorder_point_total	INT,
last_updated_date	DATE,
is_current STRING)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_thresholds/*'],
  field_delimiter = ',', skip_leading_rows=1);"""

run_bigquery_sql(LOAD_stock_thresholds_SQL)


### Stock Transfer Orders & Stock Transfer Order Items

In [ ]:
LOAD_stock_transfer_orders_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_transfer_orders(stock_transfer_order_id STRING,
    from_location_id STRING,
    to_location_id STRING,
    status STRING,
    vehicle_id STRING,
    reference_id STRING,
    reference_id_type STRING,
    transfer_type STRING,
    transfer_order_date DATE,
    fulfillment_date DATE)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_transfer_orders/*'],
  field_delimiter = ',', skip_leading_rows=1);"""


run_bigquery_sql(LOAD_stock_transfer_orders_SQL)

In [ ]:
LOAD_stock_transfer_order_items_SQL=f"""
LOAD DATA OVERWRITE capstone_ds.stock_transfer_order_items(stock_transfer_order_id STRING,
    item_number STRING,
    omni_item_id STRING,
    quantity INT)
FROM FILES (
  format = 'csv',
  uris = ['gs://{DATA_INGESTION_BUCKET}/fridge_tabular_data/stock_transfer_order_items/*'],
  field_delimiter = ',', skip_leading_rows=1);"""


run_bigquery_sql(LOAD_stock_transfer_order_items_SQL)

## 4. [BigQuery] Create stored procedures

### Log Agent Activity

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE PROCEDURE `capstone_ds.log_agent_activity`(
  p_item_number STRING,p_omni_item_id STRING,p_activity_type STRING,p_activity_detail STRING,p_execution_date DATE,p_executed_by STRING
)
BEGIN


  INSERT INTO `capstone_ds.agent_activity_log`(item_number,omni_item_id,activity_type,activity_detail,execution_date,executed_by)
  select p_item_number,p_omni_item_id,p_activity_type,p_activity_detail,p_execution_date,p_executed_by;


  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      v_forecast_datetime,
      'log_agent_activity',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);

END;

### Log Demand Signal

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE PROCEDURE `capstone_ds.log_demand_signal`(p_item_number STRING, p_omni_item_id  STRING, p_signal_indicator  STRING, p_comment  STRING, p_notified_by  STRING)
BEGIN

  DECLARE v_signal_datetime DATETIME DEFAULT CURRENT_DATETIME;

  BEGIN

    UPDATE capstone_ds.demand_signal_log
    SET is_active=false
    WHERE 1=1;

    INSERT INTO capstone_ds.demand_signal_log(item_number, omni_item_id, signal_indicator, comment, notified_by, signal_datetime,is_active)
    select p_item_number, p_omni_item_id, p_signal_indicator, p_comment, p_notified_by,v_signal_datetime,true;

  EXCEPTION WHEN ERROR THEN
    INSERT INTO capstone_ds.procedure_error_log(error_time, procedure_name, error_message)
    VALUES (v_forecast_datetime, 'log_demand_signal', @@error.message);
    RAISE USING MESSAGE = @@error.message;
  END;
END;

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

--CALL capstone_ds.log_demand_signal('LRDCS2603S','LRDCS2603S','SURGE','This product was endorsed by a celebrity on social media and there is a demand surge noticed with our competitors','MARKET_INTELLIGENCE_AGENT');

### Generate Demand Forecast

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE PROCEDURE `capstone_ds.run_demand_forecast`(p_invoker STRING)
BEGIN
  -- 1. Declare and initiative variables
  DECLARE v_forecast_id STRING DEFAULT GENERATE_UUID();
  DECLARE v_forecast_datetime DATETIME DEFAULT CURRENT_DATETIME;

  -- 2. Save existing forecast to history table before overwriting
  -- Ensure demand_forecast_history exists and has a compatible schema
  INSERT INTO `capstone_ds.demand_forecast_history`
  SELECT * FROM `capstone_ds.demand_forecast`;

  TRUNCATE TABLE capstone_ds.demand_forecast;

  -- 3. Run the latest forecast
  -- Overwrites the current forecast table with fresh 30-day predictions
  BEGIN
    INSERT INTO capstone_ds.demand_forecast
    SELECT DISTINCT v_forecast_id,v_forecast_datetime as forecast_run_time,p_invoker as generated_by,'' as item_number, '' as item_name, location_id ,omni_item_id,
    forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
    FROM
      AI.FORECAST(
        (
          SELECT *
          FROM capstone_ds.vw_aggr_sales_by_item
        ),
        horizon => 30,
        confidence_level => 0.95,
        timestamp_col => 'transaction_date',
        data_col => 'total_quantity',
        id_cols => ['location_id', 'omni_item_id']
        )
      ;

    -- 4. Enrich the forecast with Product Master details
    -- Joins on item IDs to fill in readable names and numbers
    MERGE `capstone_ds.demand_forecast` AS DF
    USING `capstone_ds.product_master` AS PM
    ON DF.omni_item_id = PM.omni_item_id
    WHEN MATCHED THEN
      UPDATE SET
        DF.item_number = PM.item_number,
        DF.item_name = PM.description;

    --5. Log activity to table
    INSERT INTO `capstone_ds.forecast_activity_log`(item_number,
    omni_item_id,forecast_id,activity_type,execution_date,executed_by)
    select null,null,v_forecast_id,'FORECAST_RUN',v_forecast_datetime,p_invoker;

    CALL capstone_ds.log_agent_activity(null,null,'FORECAST_RUN',concat('Forecast ID - ',v_forecast_id),Date(v_forecast_datetime),p_invoker);



  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      v_forecast_datetime,
      'run_demand_forecast',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);
  END;

END;

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

--CALL capstone_ds.run_demand_forecast('BATCH_PROCESS');

### Update Demand Forecast

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


CREATE OR REPLACE PROCEDURE `capstone_ds.update_item_demand_forecast`(p_omni_item_id STRING, p_adjustment_type STRING, p_invoker STRING)
BEGIN
  -- 1. Declare and initialize variables
  DECLARE v_forecast_batch_id STRING DEFAULT GENERATE_UUID();
  DECLARE v_forecast_datetime DATETIME DEFAULT CURRENT_DATETIME;

  --2. Get the multipler for the demand adjustment
  DECLARE v_demand_multiplier FLOAT64;

  SET v_demand_multiplier = (
    SELECT (CASE WHEN p_adjustment_type='SURGE' THEN demand_surge_multiplier ELSE demand_slump_multiplier END)
    FROM `capstone_ds.forecast_override_configs`
    WHERE omni_item_id = p_omni_item_id
  );

  BEGIN
    -- 3. Archive only the product being updated
    -- This keeps your history clean and specific
    INSERT INTO `capstone_ds.demand_forecast_history`
    SELECT * FROM `capstone_ds.demand_forecast`
    WHERE (omni_item_id = p_omni_item_id);

    -- 4. Update the existing forecast to accomodate the demand surge
    UPDATE `capstone_ds.demand_forecast`
    SET generated_by = 'AGENT_OVERRIDE',
        forecast_value = forecast_value * v_demand_multiplier,
        forecast_id = v_forecast_batch_id,
        forecast_run_time = v_forecast_datetime
    WHERE omni_item_id = p_omni_item_id;

    --5. Log the activity into the activty log table
    INSERT INTO `capstone_ds.forecast_activity_log`(item_number,
    omni_item_id,forecast_id,activity_type,execution_date,executed_by)
    select null,p_omni_item_id,v_forecast_batch_id,'FORECAST_UPDATE',v_forecast_datetime,p_invoker;

    CALL capstone_ds.log_agent_activity(null,null,'FORECAST_UPDATE',concat('Forecast ID - ',v_forecast_batch_id),Date(v_forecast_datetime),p_invoker);


  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (error_time, procedure_name, error_message)
    VALUES (v_forecast_datetime, 'update_item_demand_forecast', @@error.message);
    RAISE USING MESSAGE = @@error.message;
  END;
END;

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

--CALL capstone_ds.update_item_demand_forecast('LRDCS2603S','SURGE','DEMAND PLANNER AGENT');


### Generate Inventory Allocation Plan

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE PROCEDURE `capstone_ds.generate_inventory_allocation_plan`(p_invoker STRING)
BEGIN

  DECLARE v_allocation_date DATE;

  SET v_allocation_date = CURRENT_DATE();

  DELETE FROM capstone_ds.stock_allocation_plan
  WHERE 1=1;


  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_allocation_date,pm.item_number,pm.omni_item_id,'NAP-IL-ST',20,v_allocation_date,true
  FROM capstone_ds.product_master pm where is_active='Y';

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_allocation_date,pm.item_number,pm.omni_item_id,'SCH-IL-ST',20,v_allocation_date,true
  FROM capstone_ds.product_master pm where is_active='Y';

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_allocation_date,pm.item_number,pm.omni_item_id,'CHI-IL-ST',20,v_allocation_date,true
  FROM capstone_ds.product_master pm where is_active='Y';

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_allocation_date,pm.item_number,pm.omni_item_id,'WHE-IL-WH',210,v_allocation_date,true
  FROM capstone_ds.product_master pm where is_active='Y';

  INSERT INTO `capstone_ds.agent_activity_log`(item_number,omni_item_id,activity_type,activity_detail,execution_date,executed_by)
  select null,null,'GENERATE_INVENTORY_ALLOCATION_PLAN','On-demand request',v_allocation_date,p_invoker;

  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      v_forecast_datetime,
      'generate_inventory_allocation_plan',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);

END;

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--CALL capstone_ds.generate_inventory_allocation_plan('BATCH_PROCESS');

### Adjust Inventory Allocation Plan

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE PROCEDURE `capstone_ds.adjust_inventory_allocation`(p_omni_item_id STRING, p_at_store_allocation INT64, p_at_warehouse_allocation INT64, p_invoker STRING)
BEGIN

  DECLARE v_new_allocation_date DATE DEFAULT CURRENT_DATE;
  DECLARE v_item_number STRING;
  DECLARE v_adjustment_detail STRING;

  SET v_item_number=(select distinct item_number from capstone_ds.stock_allocation_plan where omni_item_id=p_omni_item_id);
  SET v_adjustment_detail = (select CONCAT('at_store_allocation=',p_at_store_allocation, ' and at_warehouse_allocation=',p_at_warehouse_allocation));

  UPDATE capstone_ds.stock_allocation_plan
  SET
    is_current=false,
    update_date=v_new_allocation_date
  WHERE omni_item_id=p_omni_item_id;

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_new_allocation_date,v_item_number,p_omni_item_id,'NAP-IL-ST',p_at_store_allocation,v_new_allocation_date,true;

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_new_allocation_date,v_item_number,p_omni_item_id,'SCH-IL-ST',p_at_store_allocation ,v_new_allocation_date,true;

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_new_allocation_date,v_item_number,p_omni_item_id,'CHI-IL-ST',p_at_store_allocation,v_new_allocation_date,true;

  INSERT INTO capstone_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date,is_current)
  SELECT v_new_allocation_date,v_item_number,p_omni_item_id,'WHE-IL-WH',p_at_warehouse_allocation*3,v_new_allocation_date,true;


  --INSERT INTO `capstone_ds.agent_activity_log`(item_number,omni_item_id,activity_type,activity_detail,execution_date,executed_by)
  --select v_item_number,p_omni_item_id,'INVENTORY_ALLOCATION_ADJUSTMENT',v_adjustment_detail,v_new_allocation_date,p_invoker;
  CALL capstone_ds.log_agent_activity(v_item_number,p_omni_item_id,'INVENTORY_ALLOCATION_ADJUSTMENT',v_adjustment_detail,v_new_allocation_date,p_invoker);



  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      v_forecast_datetime,
      'adjust_inventory_allocation',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);

END;

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

--CALL capstone_ds.adjust_inventory_allocation('LRDCS2603S',50,40,'TESTER');

### Generate Stock Transfer Order

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

-- This is never called anywhere and was used by the author for generating the initial base data
CREATE OR REPLACE PROCEDURE `capstone_ds.generate_stock_transfer_order`(p_omni_item_id STRING, p_invoker STRING)



BEGIN

  DECLARE v_stock_transfer_order_date DATE DEFAULT CURRENT_DATE();
  DECLARE v_stock_transfer_order_id_new STRING DEFAULT GENERATE_UUID();

  -- 1. Initial Deficit Calculation
  CREATE OR REPLACE TEMP TABLE replenishment_calc AS
  WITH current_inventory AS (
    SELECT location_id, omni_item_id, SUM(quantity_on_hand) as physical_qty
    FROM `capstone_ds.stock_master_location`
    WHERE stock_date = (SELECT MAX(stock_date) FROM `capstone_ds.stock_master_location`)
      AND omni_item_id = p_omni_item_id
    GROUP BY 1, 2
  ),
  warehouse_supply AS (
    SELECT physical_qty as wh_starting_stock
    FROM current_inventory
    WHERE location_id = 'WHE-IL-WH'
  ),
  store_demands AS (
    SELECT
      p.location_id,
      p.omni_item_id,
      p.quantity as target_allocation,
      COALESCE(i.physical_qty, 0) as current_on_hand,
      GREATEST(0, p.quantity - COALESCE(i.physical_qty, 0)) as qty_requested,
      CASE
        WHEN p.location_id = 'NAP-IL-ST' THEN 'Nomad'
        WHEN p.location_id = 'CHI-IL-ST' THEN 'Dart'
        WHEN p.location_id = 'SCH-IL-ST' THEN 'RustyRose'
      END as assigned_vehicle
    FROM `capstone_ds.stock_allocation_plan` p
    LEFT JOIN current_inventory i ON p.location_id = i.location_id
    WHERE p.omni_item_id = p_omni_item_id
      AND p.is_current = true
      AND p.location_id IN ('NAP-IL-ST', 'CHI-IL-ST', 'SCH-IL-ST')
  )
  SELECT
    s.*,
    w.wh_starting_stock,
    LEAST(1.0, SAFE_DIVIDE(w.wh_starting_stock, SUM(s.qty_requested) OVER())) as distribution_factor
  FROM store_demands s, warehouse_supply w;

  -- 2. Apply Floor rounding and calculate the "Rounding Residue"
  -- We identify how many units are 'left on the dock' due to rounding down
  CREATE OR REPLACE TEMP TABLE rounding_stage AS
  SELECT
    *,
    FLOOR(qty_requested * distribution_factor) as base_transfer_qty,
    -- Ranking stores by who has the biggest remaining gap after the base transfer
    -- This determines who gets the 'leftover' units first
    RANK() OVER(ORDER BY (qty_requested - FLOOR(qty_requested * distribution_factor)) DESC, location_id) as priority_rank
  FROM replenishment_calc;

  -- 3. Final Transfer Calculation (Distributing the Residue)
  CREATE OR REPLACE TEMP TABLE final_transfers AS
  WITH residue_calc AS (
    SELECT
      wh_starting_stock - SUM(base_transfer_qty) OVER() as leftover_units
    FROM rounding_stage
    LIMIT 1
  )
  SELECT
    rs.*,
    -- If your priority rank is within the count of leftover units, you get +1 fridge
    CASE
      WHEN rs.priority_rank <= (SELECT leftover_units FROM residue_calc) THEN rs.base_transfer_qty + 1
      ELSE rs.base_transfer_qty
    END as final_transfer_qty
  FROM rounding_stage rs;

  -- 4. Execute the Inserts (Same Loop Logic)
  FOR record IN (SELECT * FROM final_transfers WHERE final_transfer_qty > 0)
  DO
    BEGIN
      DECLARE current_po_id STRING DEFAULT GENERATE_UUID();

      INSERT INTO `capstone_ds.stock_transfer_orders` (
        transfer_order_date, stock_transfer_order_id, from_location_id, to_location_id,
        status, vehicle_id, reference_id, reference_id_type, fulfillment_date, transfer_type
      )
      VALUES (
        v_stock_transfer_order_date, v_stock_transfer_order_id_new, 'WHE-IL-WH', record.location_id,
        'PENDING', record.assigned_vehicle, 'N/A', 'ALLOCATION_OVERRIDE',
        DATE_ADD(v_stock_transfer_order_date, INTERVAL 1 DAY), 'INCREMENTAL_STOCK_DELIVERY_FROM_WAREHOUSE'
      );

      INSERT INTO `capstone_ds.stock_transfer_order_items` (
        stock_transfer_order_id, item_number, omni_item_id, quantity
      )
      SELECT v_stock_transfer_order_id_new, pm.item_number, record.omni_item_id, cast(round(record.final_transfer_qty,0) as int64)
      FROM `capstone_ds.product_master` pm WHERE pm.omni_item_id = record.omni_item_id;
    END;

      --INSERT INTO `capstone_ds.agent_activity_log`(item_number,omni_item_id,activity_type,activity_detail,execution_date,executed_by)
      --select null,p_omni_item_id,'STOCK_TRANSFER_ORDER','On-demand-based-on-allocation-change',v_transfer_order_date,p_invoker;
      CALL capstone_ds.log_agent_activity(null,p_omni_item_id,'STOCK_TRANSFER_ORDER',Concat('On-demand-for-',p_omni_item_id),v_stock_transfer_order_date,p_invoker);

  END FOR;



  -- 5. Results Summary
  SELECT
    v_stock_transfer_order_date as order_date,v_stock_transfer_order_id_new as order_id,p_omni_item_id,
    location_id, assigned_vehicle, qty_requested, final_transfer_qty,
    (wh_starting_stock - SUM(final_transfer_qty) OVER()) as wh_remaining_after_run
  FROM final_transfers
  WHERE final_transfer_qty > 0;



  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      CURRENT_DATE(),
      'generate_stock_transfer_order',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);

END;

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--CALL capstone_ds.generate_stock_transfer_order('LRDCS2603S','Anagha');

### Generate Stock Purchase Order

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE PROCEDURE `capstone_ds.generate_stock_purchase_order`(p_omni_item_id STRING, p_invoker STRING)
BEGIN

  DECLARE v_po_id STRING DEFAULT GENERATE_UUID();
  DECLARE v_supplier_id STRING;
  DECLARE v_order_date,v_expected_date,v_received_date DATE;

  SET v_order_date = CURRENT_DATE();
  SET v_expected_date = DATE_ADD(v_order_date, INTERVAL 6 DAY);
  SET v_received_date = null;
  SET v_supplier_id = (select distinct UPPER(supplier_id) from capstone_ds.product_suppliers where omni_item_id=p_omni_item_id);


  INSERT INTO `capstone_ds.stock_purchase_orders` (po_id,supplier_id,order_date,expected_date,received_date,total_cost,order_status,order_comment)
  select v_po_id,v_supplier_id,v_order_date,v_expected_date,v_received_date,0,'PENDING','PURCHASE_ORDER_INCREMENTAL_STOCK';

  INSERT INTO `capstone_ds.stock_purchase_order_items` (po_id,line_item_id,item_number,omni_item_id,quantity_ordered,unit_price)
  select v_po_id,GENERATE_UUID(),pm.item_number, p_omni_item_id, 180,pm.price
  from capstone_ds.product_master pm
  where pm.omni_item_id=p_omni_item_id;

  MERGE capstone_ds.stock_purchase_orders PO
  USING (
    SELECT po_id, SUM(quantity_ordered * unit_price) AS total_cost
    FROM capstone_ds.stock_purchase_order_items
    WHERE po_id=v_po_id
    GROUP BY po_id
  ) AS POI
  ON PO.po_id = POI.po_id
  WHEN MATCHED THEN
    UPDATE SET total_cost = POI.total_cost;

  INSERT INTO `capstone_ds.agent_activity_log`(item_number,omni_item_id,activity_type,activity_detail,execution_date,executed_by)
  select null,p_omni_item_id,'STOCK_PURCHASE_ORDER','On-demand',v_order_date,p_invoker;

  SELECT
    po.supplier_id,po.order_date,po.expected_date,poi.omni_item_id,poi.quantity_ordered,po.total_cost
  FROM capstone_ds.stock_purchase_orders po join capstone_ds.stock_purchase_order_items poi on po.po_id=poi.po_id
  WHERE po.po_id=v_po_id;


  EXCEPTION WHEN ERROR THEN
    INSERT INTO `capstone_ds.procedure_error_log` (
      error_time,
      procedure_name,
      error_message,
      statement_text
    )
    VALUES (
      v_order_date,
      'generate_stock_purchase_order',
      @@error.message,          -- The actual error message
      @@error.statement_text    -- The specific SQL that failed
    );

    RAISE USING MESSAGE = FORMAT("Procedure failed: %s", @@error.message);

END;



In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

--CALL capstone_ds.generate_stock_purchase_order('LRDCS2603S','BATCH_PROCESS');

## 5. Run initial set up routines

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--Generate 30 day forecast
CALL capstone_ds.run_demand_forecast('BATCH_PROCESS');

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--Generate inventory allocation plan
CALL capstone_ds.generate_inventory_allocation_plan('BATCH_PROCESS');

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--Log a demand signal
CALL capstone_ds.log_demand_signal('LRDCS2603S','LRDCS2603S','SURGE','This product was endorsed by a celebrity on social media and there is a demand surge noticed with our competitors','MARKET_INTELLIGENCE_AGENT');

## 6. [Dataplex] Data Insights Scans

#### [Ignore this] Incremental table additions

In [ ]:
# This function is when you want run incrementally for select tables added
"""
def run_targeted_table_scans(CORE_DATASET_ID: str, METADATA_DATASET_ID: str, METADATA_BUCKET: str, GROUNDING_FILE_NAME: str, TABLE_LIST: list = None):
    """
    Runs asynchronous table scans for a specific list of tables,
    then refreshes the dataset scan and grounding file.
    """
    start_time = time.time()
    token = get_access_token()

    # Use supplied list if available, otherwise fetch all tables in the dataset
    tables_to_scan = TABLE_LIST if TABLE_LIST else fetch_list_of_tables_in_dataset(CORE_DATASET_ID)

    print(f"🚀 Starting targeted scans for {len(tables_to_scan)} tables in {CORE_DATASET_ID}...")

    # 1. Trigger Async Table Documentation Scans
    active_jobs = {}
    for table_nm in tables_to_scan:
        scan_id = f"cs-{table_nm.replace('_','-')}-table-documentation-scan"

        # Ensure the scan is created and patched for the UI
        create_scan_synchronous(token, CORE_DATASET_ID, scan_id, "DATA_DOCUMENTATION_SCAN", table_nm)
        patch_source_table_with_labels(token, CORE_DATASET_ID, "DATA_DOCUMENTATION_SCAN", scan_id, table_nm)

        # Trigger the run
        headers = {"Authorization": f"Bearer {token}"}
        resp = requests.post(get_scan_api_endpoint("RUN_SCAN", scan_id), headers=headers)
        if resp.status_code == 200:
            active_jobs[scan_id] = resp.json()["job"]["name"]
            print(f"📡 Triggered: {table_nm}")

    # 2. Monitor Table Scans
    completed = set()
    while len(completed) < len(active_jobs):
        for s_id, j_name in active_jobs.items():
            if s_id in completed: continue
            status = requests.get(f"https://dataplex.googleapis.com/v1/{j_name}",
                                  headers={"Authorization": f"Bearer {token}"}).json().get("state")
            if status in ["SUCCEEDED", "FAILED"]:
                if status == "SUCCEEDED":
                    persist_documentation_scan_table_metadata(token, s_id)
                completed.add(s_id)
                print(f"✨ Table Job Finished: {s_id}")
        if len(completed) < len(active_jobs):
            time.sleep(30)

    # 3. Run Dataset Documentation Scan
    print("\n📂 Dataset Scan...")
    ds_scan_id = sanitize_string_with_hyphens(f"{CORE_DATASET_ID}-dataset-documentation-scan")
    run_scan_synchronous(token, ds_scan_id)

    # 4. Final Persistence & Grounding File Creation

    print("\n💾 Persisting Knowledge Engine results...")
    persist_dataplex_scan_output_to_bq_tables(CORE_DATASET_ID, METADATA_DATASET_ID)

    print(f"\n📝 Generating revised grounding file: {GROUNDING_FILE_NAME}")
    generate_metadata_grounding_file(CORE_DATASET_ID, METADATA_DATASET_ID, METADATA_BUCKET, GROUNDING_FILE_NAME)

    print(f"\n🏁 Targeted Workflow Complete! Runtime: {(time.time() - start_time) / 60:.2f} mins.")

"""

In [ ]:
# Run scans for specific set of tables - this is for incremental tables or views added | Comment out the c
#REDO_LIST=["vw_inventory_aging","vw_sales_history","vw_stock_master","vw_stock_master_location","vw_stock_movement","vw_stock_movement_expanded","vw_stock_movement_summary","vw_stock_purchase_orders","vw_stock_reconciliation","vw_stock_reorder_points","vw_stock_transfer_aggr","vw_stock_transfer_orders","vw_suggested_reorder","vw_supplier_fill_rate","vw_weighted_lead_time"]
#run_targeted_table_scans("capstone_ds", "capstone_metadata_ds", METADATA_BUCKET, "captone_database_metadata_grounding.md",REDO_LIST)

### 5.1. Utils

In [ ]:
import requests
import json
import time
import re
import itertools
import pandas as pd
import google.auth
import google.auth.transport.requests
from google.cloud import bigquery, storage
from google.api_core.exceptions import NotFound
from urllib.parse import urlencode

# --- 1. CORE UTILITIES & AUTHENTICATION ---

def sanitize_string_with_hyphens(input_string):
    """Converts string to lowercase and replaces non-alphanumeric with hyphens."""
    processed_string = input_string.lower()
    return re.sub(r'[^a-z0-9]', '-', processed_string)

def get_access_token():
    """Generates an access token using ADC."""
    try:
        credentials, project = google.auth.default(scopes=SCOPES)
        request = google.auth.transport.requests.Request()
        credentials.refresh(request)
        return credentials.token
    except Exception as e:
        print(f"❌ Error generating access token: {e}")
        return None

def get_bq_client():
    """Initializes and returns a BigQuery client."""
    try:
        return bigquery.Client(project=PROJECT_ID)
    except Exception as e:
        print(f"❌ Failed to create BigQuery client: {e}")
        return None

# --- 2. API HELPERS & ENDPOINTS ---

def get_scan_api_endpoint(scan_operation_type, scan_id):
    """Returns the Dataplex API endpoint based on operation."""
    if scan_operation_type == "CREATE_SCAN":
        return f"{DATA_SCAN_API_CREATE_ENDPOINT_PREFIX}{scan_id}"
    return f"{DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX}{scan_id}:run"

def generate_scan_request_body(dataset_id, scan_id, scan_type, source_table_nm=""):
    """Generates the JSON payload for Dataplex."""
    if scan_type == "DATA_DOCUMENTATION_SCAN":
        return {
            "displayName": f"{scan_id}",
            "type": "DATA_DOCUMENTATION",
            "data": {"resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}/tables/{source_table_nm}"},
            "dataDocumentationSpec": {},
            "executionSpec": {"trigger": {"onDemand": {}}}
        }
    elif scan_type == "DATA_KNOWLEDGE_ENGINE_SCAN":
        return {
            "displayName": f"{scan_id}",
            "type": "DATA_DOCUMENTATION",
            "dataDocumentationSpec": {},
            "dataDocumentationResult": {},
            "data": {"resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}"},
            "executionSpec": {"trigger": {"onDemand": {}}}
        }

def generate_patch_label_request_body(scan_id):
    """Labels to link programmatic scans to the GCP Console UI."""
    return {
        "labels": {
            "dataplex-data-documentation-published-scan": f"{scan_id}",
            "dataplex-data-documentation-published-project": f"{PROJECT_ID}",
            "dataplex-data-documentation-published-location": f"{LOCATION}"
        }
    }

def get_scan_results(token, scan_id):
    """Retrieves full scan results with documentation details."""
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}?view=FULL"
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(url) # Added for response handling logic
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

# --- 3. BIGQUERY RESOURCE & TABLE MANAGEMENT ---

def update_bigquery_metadata(project_id, dataset_id, new_description, table_id=None):
    """Updates descriptions directly on BigQuery resources."""
    client = bigquery.Client(project=project_id)
    try:
        if table_id:
            table_ref = client.dataset(dataset_id).table(table_id)
            table = client.get_table(table_ref)
            table.description = new_description
            client.update_table(table, ["description"])
            print(f"✅ BQ Update: Table '{table_id}' description saved.")
        else:
            dataset_ref = client.dataset(dataset_id)
            dataset = client.get_dataset(dataset_ref)
            dataset.description = new_description
            client.update_dataset(dataset, ["description"])
            print(f"✅ BQ Update: Dataset '{dataset_id}' description saved.")
    except Exception as e:
        print(f"⚠️ Metadata update failed: {e}")

def truncate_bigquery_table(bq_table_uri):
    """Clears a BigQuery table before re-populating."""
    client = bigquery.Client()
    query_job = client.query(f"TRUNCATE TABLE `{bq_table_uri}`")
    query_job.result()

def write_dict_to_bigquery(bq_table_uri, data_to_insert):
    """Inserts a row (dict) into a BigQuery table."""
    client = bigquery.Client()
    errors = client.insert_rows_json(bq_table_uri, [data_to_insert])
    if errors: print(f"❌ BQ Insert Error: {errors}")

def fetch_list_of_tables_in_dataset(dataset_id):
    """Lists all table IDs within a dataset."""
    client = get_bq_client()
    tables = client.list_tables(dataset_id)
    return [table.table_id for table in tables]

# --- 4. SCAN EXECUTION & PATCHING ---

def create_scan_synchronous(access_token, dataset_id, scan_id, scan_type, source_table_nm=""):
    """Creates a scan and waits for completion."""
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    body = generate_scan_request_body(dataset_id, scan_id, scan_type, source_table_nm)
    url = get_scan_api_endpoint("CREATE_SCAN", scan_id)
    resp = requests.post(url, headers=headers, json=body)
    if resp.status_code != 409:
        resp.raise_for_status()
        poll_data_scan_operation("CREATE_SCAN", resp.json()["name"], access_token)

def patch_source_table_with_labels(access_token, dataset_id, scan_type, scan_id, source_table_nm):
    """Applies labels to a BQ table."""
    url = f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/datasets/{dataset_id}/tables/{source_table_nm}"
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    requests.patch(url, headers=headers, json=generate_patch_label_request_body(scan_id))

def patch_source_dataset_with_labels(access_token, dataset_id, scan_type, scan_id):
    """Applies labels to a BQ dataset."""
    url = f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/datasets/{dataset_id}"
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    requests.patch(url, headers=headers, json=generate_patch_label_request_body(scan_id))

def run_scan_synchronous(access_token, scan_id):
    """Triggers and polls a scan sequentially."""
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    url = get_scan_api_endpoint("RUN_SCAN", scan_id)
    resp = requests.post(url, headers=headers, json={})
    resp.raise_for_status()
    return poll_data_scan_operation("RUN_SCAN", resp.json()['job']['name'], access_token)

def poll_data_scan_operation(op_type, op_name, access_token):
    """Wait for Dataplex operations to complete."""
    headers = {"Authorization": f"Bearer {access_token}"}
    url = f"https://dataplex.googleapis.com/v1/{op_name}"
    while True:
        resp = requests.get(url, headers=headers).json()
        if op_type == "CREATE_SCAN" and resp.get("done"): return resp
        if op_type == "RUN_SCAN" and resp.get("state") in ["SUCCEEDED", "FAILED", "CANCELLED"]: return resp
        time.sleep(30)

# --- 5. METADATA PERSISTENCE & GROUNDING ---

def persist_dataplex_scan_output_to_bq_tables(source_ds, meta_ds):
    """Updates Dataset description and populates metadata tables for grounding."""
    token = get_access_token()
    ds_scan_id = sanitize_string_with_hyphens(f"{source_ds}-dataset-documentation-scan")

    try:
        # Fetch Dataset-level results
        res = get_scan_results(token, ds_scan_id).get("dataDocumentationResult", {}).get("datasetResult", {})

        # 1. Update Native Dataset Description
        dataset_overview = res.get("overview", "")
        if dataset_overview:
            update_bigquery_metadata(PROJECT_ID, source_ds, dataset_overview)

            # Log to metadata table for grounding
            truncate_bigquery_table(f"{PROJECT_ID}.{meta_ds}.dataset_description")
            write_dict_to_bigquery(f"{PROJECT_ID}.{meta_ds}.dataset_description",
                                   {"dataset_description": dataset_overview})

        # 2. Update Table Relationships in Metadata Table
        truncate_bigquery_table(f"{PROJECT_ID}.{meta_ds}.dataset_table_relationships")
        for rel in res.get("schemaRelationships", []):
            write_dict_to_bigquery(f"{PROJECT_ID}.{meta_ds}.dataset_table_relationships", {
                "table_1": rel.get("leftSchemaPaths", {}).get("tableFqn", "").split("/")[-1],
                "table_1_column": rel.get("leftSchemaPaths", {}).get("paths", [""])[0],
                "table_2": rel.get("rightSchemaPaths", {}).get("tableFqn", "").split("/")[-1],
                "table_2_column": rel.get("rightSchemaPaths", {}).get("paths", [""])[0],
                "join_type": rel.get("type", "JOIN").replace("SCHEMA_JOIN", "JOIN")
            })

        # 3. Populate Metadata Tracking Tables
        truncate_bigquery_table(f"{PROJECT_ID}.{meta_ds}.table_descriptions")
        truncate_bigquery_table(f"{PROJECT_ID}.{meta_ds}.table_column_descriptions")

        tables = fetch_list_of_tables_in_dataset(source_ds)
        for table_nm in tables:
            t_scan_id = f"cs-{table_nm.replace('_','-')}-table-documentation-scan"
            try:
                t_res = get_scan_results(token, t_scan_id).get("dataDocumentationResult", {}).get("tableResult", {})

                # Log table overview to metadata table
                t_desc = t_res.get("overview", "")
                if t_desc:
                    write_dict_to_bigquery(f"{PROJECT_ID}.{meta_ds}.table_descriptions",
                                           {"name": table_nm, "description": t_desc})

                # Log column details to metadata table
                for field in t_res.get("schema", {}).get("fields", []):
                    if field.get("description"):
                        write_dict_to_bigquery(f"{PROJECT_ID}.{meta_ds}.table_column_descriptions", {
                            "table_name": table_nm,
                            "column_name": field["name"],
                            "column_description": field["description"]
                        })
            except Exception:
                continue

    except Exception as e:
        print(f"❌ Error during full metadata persistence: {e}")

def generate_metadata_grounding_file(source_ds, meta_ds, bucket, file_nm):
    """Builds the agent grounding markdown and uploads to GCS."""
    client = bigquery.Client()
    content = f"Metadata for dataset: {source_ds}\n\n"

    # Helper to append table content
    def append_table(table_name, header):
        nonlocal content
        df = client.query(f"SELECT * FROM `{PROJECT_ID}.{meta_ds}.{table_name}`").to_dataframe()
        content += f"### {header}\n{df.to_markdown(index=False)}\n\n"

    append_table("dataset_description", "Dataset Overview")
    append_table("table_descriptions", "Table Descriptions")
    append_table("dataset_table_relationships", "Table Relationships")
    append_table("table_column_descriptions", "Column Details")

    storage.Client().bucket(bucket).blob(file_nm).upload_from_string(content)
    print(f"📝 Grounding file '{file_nm}' uploaded to {bucket}.")

# --- 6. ORCHESTRATION ---

def execute_table_documentation_scan_for_a_dataset(DATASET_ID, batch_size=10):
    """Triggers and polls table scans in controlled batches to avoid 429 errors."""
    token = get_access_token()
    tables = fetch_list_of_tables_in_dataset(DATASET_ID)

    # Logic to process the list in chunks of batch_size
    for i in range(0, len(tables), batch_size):
        batch = tables[i:i + batch_size]
        active_scans_set = {}

        print(f"\n📦 Processing Batch {(i // batch_size) + 1} ({len(batch)} tables)")

        # 1. Trigger Phase for current batch
        for table_nm in batch:
            if table_nm == 'vw_inventory_aging_report': # Skip problematic table
                continue

            s_id = f"cs-{table_nm.replace('_','-')}-table-documentation-scan"
            create_scan_synchronous(token, DATASET_ID, s_id, "DATA_DOCUMENTATION_SCAN", table_nm)
            patch_source_table_with_labels(token, DATASET_ID, "DATA_DOCUMENTATION_SCAN", s_id, table_nm)

            headers = {"Authorization": f"Bearer {token}"}
            resp = requests.post(get_scan_api_endpoint("RUN_SCAN", s_id), headers=headers)

            # 1-second delay between triggers within the batch
            time.sleep(1)

            if resp.status_code == 200:
                active_scans_set[s_id] = resp.json()["job"]["name"]
                print(f"📡 Triggered async scan: {table_nm}")

        # 2. Polling Phase for current batch
        completed_scans_set = set()
        print(f"⏳ Polling {len(active_scans_set)} scans in this batch...")

        while len(completed_scans_set) < len(active_scans_set):
            for s_id, j_name in active_scans_set.items():
                if s_id in completed_scans_set:
                    continue

                status_resp = requests.get(
                    f"https://dataplex.googleapis.com/v1/{j_name}",
                    headers={"Authorization": f"Bearer {token}"}
                ).json()

                status = status_resp.get("state")
                if status in ["SUCCEEDED", "FAILED", "CANCELLED"]:
                    if status == "SUCCEEDED":
                        print(f"✨ Scan job Finished: {s_id}")
                        persist_documentation_scan_table_metadata(token, s_id)
                    else:
                        print(f"❌ Scan job {status}: {s_id}")

                    completed_scans_set.add(s_id)

            if len(completed_scans_set) < len(active_scans_set):
                time.sleep(15) # Wait before next status check

        time.sleep(180) # Sleep some after each batch

    print("\n✅ All table batches completed.")

def execute_dataset_documentation_scan(DATASET_ID):
    """Knowledge Engine scan for dataset-level insights."""
    print(f"Executing dataset documentation scan")
    token = get_access_token()
    s_id = sanitize_string_with_hyphens(f"{DATASET_ID}-dataset-documentation-scan")
    create_scan_synchronous(token, DATASET_ID, s_id, "DATA_KNOWLEDGE_ENGINE_SCAN")
    patch_source_dataset_with_labels(token, DATASET_ID, "DATA_KNOWLEDGE_ENGINE_SCAN", s_id)
    run_scan_synchronous(token, s_id)

def persist_documentation_scan_table_metadata(token, scan_id):
    """
    Writes the documentation results (Overview and Column descriptions)
    directly back to the native BigQuery source table metadata.
    """
    print("====================================================")
    print(f"Attempting to persist scan results from {scan_id}")
    # 1. Fetch full results from Dataplex
    res = get_scan_results(token, scan_id)

    if res:

      # 2. Extract the BigQuery Resource path
      # Format: //bigquery.googleapis.com/projects/PROJECT/datasets/DATASET/tables/TABLE
      parts = res["data"]["resource"].split("/")
      project_id, dataset_id, table_nm = parts[4], parts[6], parts[8]
      print(f"Table name: {table_nm}")

      # 3. Initialize BQ Client and get the Table object
      client = bigquery.Client(project=project_id)
      table_ref = client.dataset(dataset_id).table(table_nm)
      table = client.get_table(table_ref)

      # 4. Extract Documentation Results
      doc_result = res.get("dataDocumentationResult", {}).get("tableResult", {})

      if doc_result:

        # Update Table-level description
        table_overview = doc_result.get("overview", "")
        if table_overview:
            table.description = table_overview

        print(f"....overview: {table_overview}")

        # Update Column-level descriptions
        col_map = {f["name"]: f.get("description", "") for f in doc_result.get("schema", {}).get("fields", [])}

        new_schema = []
        for field in table.schema:
            # If the AI provided a description for this column, use it; else keep current
            updated_desc = col_map.get(field.name, field.description)
            new_schema.append(bigquery.SchemaField(
                name=field.name,
                field_type=field.field_type,
                mode=field.mode,
                description=updated_desc,
                fields=field.fields
            ))

        # 5. Commit the changes back to BigQuery
        table.schema = new_schema
        try:
          client.update_table(table, ["description", "schema"])
          print(f"✅ Source Table Metadata Updated: {dataset_id}.{table_nm}")
        except:
          print(f"Something went wrong while persisting metadata from {scan_id}")

      else:
        print(f"doc_result for {scan_id} did not return a value")

    else:
      print(f"Fetching scan results from {scan_id} failed")

def run_scans_and_persist_metadata_to_file(CORE_DATASET_ID: str, METADATA_DATASET_ID: str, METADATA_BUCKET: str, GROUNDING_FILE_NAME: str):
    """Main execution flow."""
    start = time.time()
    print(f"Starting Data Insights for the entire dataset now: {start}")

    print("Step 1: Create and execute table documentation scans async and poll for completion of all scans")
    execute_table_documentation_scan_for_a_dataset(CORE_DATASET_ID)

    print("Step 2: Create and execute dataset documentation scan synchronously")
    execute_dataset_documentation_scan(CORE_DATASET_ID)

    print("Step 3: Persist dataset documentation scans results to metadata dataset")
    persist_dataplex_scan_output_to_bq_tables(CORE_DATASET_ID, METADATA_DATASET_ID)

    print("Step 4: Persist grounding file to GCS")
    generate_metadata_grounding_file(CORE_DATASET_ID, METADATA_DATASET_ID, METADATA_BUCKET, GROUNDING_FILE_NAME)

    print(f"\n🏁 Complete! Total Runtime: {(time.time() - start) / 60:.2f} mins.")

### 5.2. Run metadata insights scans & persist insights to file in GCS for agentic grounding in subsequent modules

In [ ]:
PROJECT_ID= ! gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
PROJECT_NAME= ! gcloud projects describe $PROJECT_ID | grep name | cut -d':' -f2 | xargs
PROJECT_NAME=PROJECT_NAME[0]

LOCATION="us-central1"
DATA_SCAN_API_CREATE_ENDPOINT_PREFIX=f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId="
DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX=(f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/")
SCOPES = ['https://www.googleapis.com/auth/cloud-platform']
BASE_URL_FOR_DATAPLEX_SCAN="https://dataplex.googleapis.com/v1"


DATA_INGESTION_BUCKET=f"capstone_stage_{PROJECT_NBR}"
METADATA_BUCKET=f"capstone_stage_{PROJECT_NBR}"
LOCATION="us-central1"

CORE_DATASET_ID="capstone_ds"
METADATA_DATASET_ID="capstone_metadata_ds"
GROUNDING_FILE_NAME="metadata/captone_database_metadata_grounding.md"


# Run scans for all tables and the dataset
run_scans_and_persist_metadata_to_file(CORE_DATASET_ID, METADATA_DATASET_ID, METADATA_BUCKET, GROUNDING_FILE_NAME)

